# Standard

In [ ]:
import numpy as np
import pandas as pd

# --- Create a synthetic dataset ---
rng = np.random.default_rng(42)

n = 20000

# Numeric features
df = pd.DataFrame({
    "age": rng.normal(40, 10, size=n),            # numeric
    "exposure": rng.uniform(0.5, 2.0, size=n),    # numeric
})

# Categorical features
regions = np.array(["north", "south", "east", "west"])
segments = np.array(["A", "B"])
df["region"] = rng.choice(regions, size=n)
df["segment"] = rng.choice(segments, size=n)

# Target (Poisson-like) with a simple generative structure
# True linear predictor (just for simulation)
lp = (
    0.1 * (df["age"] - 40) +         # effect of age centered
    0.5 * (df["segment"] == "B") +   # segment B higher
    0.2 * (df["region"] == "south")  # south slightly higher
)

mu = np.exp(lp) * df["exposure"]    # mean proportional to exposure
y = rng.poisson(mu) / df["exposure"]

# Optional sample weights (e.g., use exposure as weight)
sample_weight = df["exposure"].values

df['y'] = y

# Behaviour

In [11]:
n = 10000

age = rng.integers(18, 80, size=n)
income = rng.normal(40000, 15000, size=n).clip(15000, 120000)
region = rng.integers(0, 4, size=n)
promo = rng.binomial(1, 0.3, size=n)
loyalty = rng.uniform(0, 1, size=n)
price = rng.uniform(5, 50, size=n)

beta_static = np.array([0.02, 0.00003, -0.15])
beta_elasticity = np.array([-0.8, -1.2])

intercept_static = -2
intercept_elastic = 1.5

eta_static = (intercept_static
              + beta_static[0]*age
              + beta_static[1]*income
              + beta_static[2]*region)

eta_elastic = (intercept_elastic
              + beta_elasticity[0]*promo
              + beta_elasticity[1]*loyalty)

logit_p = eta_static + price * eta_elastic
p = 1 / (1 + np.exp(-logit_p))

y =  rng.binomial(1, p)

X = pd.DataFrame({
    "age" :age,
    "income" :income,
    "region" : region,
    "promo" :promo,
    "loyalty" : loyalty,
    "price" :price
})
sample_weight = np.ones(n)

X["y"] = y
X["w"] = sample_weight